In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError("Could not locate workspace root from WORKSPACE_ROOT or the current working directory.")

WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
PAPER_DIR = paths["paper_dir"]
SHARED_ASSETS_DIR = paths["shared_assets"]

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

workspace_root: /Users/sallz/allzen/damadi/research/historical-continuity/workspace
shared_assets_root: /Users/sallz/allzen/damadi/research/shared-assets


# Tutorial: Prepare Average Citations Dataset

Audience:
- Anyone turning the manuscript companion into a reusable repo and wanting to inspect the citation-ranking pipeline step by step.

Prerequisites:
- Install the Python dependencies from `../config/requirements.txt`.
- Use the local canonical parquet files already stored under `../data/`.

Learning goals:
- By the end, the reader can rebuild `avg_citations_result.parquet` directly from the canonical local paper and metrics tables.


## Outline

1. Locate the companion config and resolve the local canonical input/output paths.
2. Load the paper table and the long-form citation metrics table.
3. Join publication years to the yearly citation rows.
4. Compute average citations per year and percentile ranks.
5. Write the derived parquet output used by the impact-figure notebook.


In [2]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import yaml
from scipy.stats import percentileofscore

CONFIG_PATH = CONFIG_DIR / "paths.yml"

def load_parquet_frame(path: Path) -> pd.DataFrame:
    """Read project parquet files through pyarrow to avoid pandas/pyarrow extension hotfix issues."""
    return pq.read_table(path).to_pandas()

CONFIG_PATH

PosixPath('/Users/sallz/allzen/damadi/research/historical-continuity/workspace/config/paths.yml')

## Step 1 - Resolve the configured paths

The config reads shared processed inputs from `shared-assets/data/processed-data/` and writes the paper-local derived parquet under `workspace/data/derived/`.


In [3]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

def resolve_path(specification):
    path = Path(str(specification)).expanduser()
    if path.is_absolute():
        return path
    return (WORKSPACE_DIR / path).resolve()

papers_path = resolve_path(config["inputs"]["papers_parquet"])
paper_metrics_long_path = resolve_path(config["inputs"]["paper_metrics_long_parquet"])
output_parquet_path = resolve_path(config["outputs"]["avg_citations_parquet"])

pd.Series(
    {
        "papers": str(papers_path),
        "paper_metrics_long": str(paper_metrics_long_path),
        "output_parquet": str(output_parquet_path),
    }
)


papers                /Users/sallz/allzen/damadi/research/shared-ass...
paper_metrics_long    /Users/sallz/allzen/damadi/research/shared-ass...
output_parquet        /Users/sallz/allzen/damadi/research/historical...
dtype: str

## Step 2 - Load the local canonical inputs

We only need publication year from the paper table and yearly citation counts from the long metrics table.


In [4]:
papers = load_parquet_frame(papers_path)[["bibcode", "year"]].rename(columns={"year": "publication_year"})
paper_metrics_long = load_parquet_frame(paper_metrics_long_path)
citation_metrics = paper_metrics_long[paper_metrics_long["metric"] == "citations"].copy()

papers.head(3)


,bibcode,publication_year
0,2015CoPhC.192..322B,2015
1,2011PhRvL.107e1301A,2011
2,2009PhRvD..79a5014A,2009


## Step 3 - Join publication year to yearly citation rows

This makes it explicit which yearly citation entries count toward the average after publication.


In [5]:
citation_metrics = citation_metrics.rename(columns={"metric_year": "year", "value": "citations"})
merged_citations = citation_metrics.merge(papers, on="bibcode", how="left")
post_publication_citations = merged_citations[merged_citations["year"] >= merged_citations["publication_year"]].copy()

post_publication_citations.head(3)


,bibcode,metric,year,citations,publication_year
0,2015CoPhC.192..322B,citations,2015,28.0,2015
1,2015CoPhC.192..322B,citations,2016,66.0,2015
2,2015CoPhC.192..322B,citations,2017,78.0,2015


## Step 4 - Compute average citations and percentile ranks

The resulting derived table is intentionally compact: `bibcode`, `avg_citations_per_year`, and `percentile`.


In [6]:
avg_citations_result = (
    post_publication_citations.groupby("bibcode")
    .agg(
        first_publication_year=("publication_year", "first"),
        latest_metric_year=("year", "max"),
        total_citations=("citations", "sum"),
    )
    .reset_index()
)

avg_citations_result["years_since_publication"] = (
    avg_citations_result["latest_metric_year"] - avg_citations_result["first_publication_year"] + 1
)
avg_citations_result["avg_citations_per_year"] = (
    avg_citations_result["total_citations"] / avg_citations_result["years_since_publication"]
)
avg_citations_values = avg_citations_result["avg_citations_per_year"]
avg_citations_result["percentile"] = avg_citations_result["avg_citations_per_year"].apply(
    lambda value: percentileofscore(avg_citations_values, value, kind="rank")
)
avg_citations_result = avg_citations_result[["bibcode", "avg_citations_per_year", "percentile"]]

avg_citations_result.head(3)


,bibcode,avg_citations_per_year,percentile
0,1912MNRAS..72..718B,0.008696,0.260155
1,1913ApJ....38..275A,0.008772,0.260787
2,1917PASP...29...91C,0.027273,0.409176


## Step 5 - Write the derived parquet output

This output stays inside `workspace/data/derived/`, which is the canonical paper-local derived-data surface for the manuscript.


In [7]:
output_parquet_path.parent.mkdir(parents=True, exist_ok=True)
avg_citations_result.to_parquet(output_parquet_path, index=False)
output_parquet_path


PosixPath('/Users/sallz/allzen/damadi/research/historical-continuity/workspace/data/derived/avg_citations_result.parquet')

# At this point, the `avg_citations_result.parquet` file is ready and can be used as input for the impact-figure notebook or any other downstream analysis.